# 12 - GPT from Scratch

**AI sin humo** - Notas personales para entender deep learning desde cero.

En los notebooks anteriores vimos attention, y cómo los Transformers combinan multi-head attention con feed-forward networks, residual connections y layer normalization. Ahora vamos a **construir un GPT completo desde cero en PyTorch**.

GPT (Generative Pre-trained Transformer) es un modelo **decoder-only**: no tiene encoder. Recibe una secuencia de tokens y predice el siguiente token. Así de simple. Esa es la única tarea: dado todo lo que vino antes, ¿cuál es la palabra más probable que sigue?

Lo que hace especial a GPT es que esa tarea tan simple — predecir la siguiente palabra — resulta ser increíblemente poderosa. Un modelo entrenado con suficientes datos para predecir la siguiente palabra termina aprendiendo gramática, hechos, razonamiento, código, y mucho más. Es la idea central detrás de ChatGPT, Claude, y todos los LLMs modernos.

En este notebook vamos a construir cada pieza, entender qué hace y por qué, y entrenar el modelo en un dataset real.

---

## Contenido

1. [Arquitectura GPT: la vista de pájaro](#arquitectura)
2. [Tokenización](#tokenizacion)
3. [Embeddings](#embeddings)
4. [Causal self-attention](#causal-attention)
5. [Feed-forward network](#ffn)
6. [Transformer block](#transformer-block)
7. [Implementación completa del GPT](#implementacion)
8. [Training loop](#training)
9. [Generación de texto](#generacion)
10. [Resumen](#resumen)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import time

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

---

<a id='arquitectura'></a>
## 1. Arquitectura GPT: la vista de pájaro

Antes de meternos en el código, veamos la arquitectura completa de un GPT. Es más simple de lo que parece:

```
Input: "El gato se sentó en el"
         │
         ▼
┌─────────────────────┐
│   Tokenización      │  texto → [45, 892, 23, 1504, 67, 45]
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│  Token Embedding    │  [45, 892, ...] → vectores de dimensión d_model
│  + Pos. Embedding   │  + información de posición (0, 1, 2, 3, ...)
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│  Transformer Block  │  ← se repite N veces
│  ┌───────────────┐  │
│  │ LayerNorm     │  │
│  │ Causal Attn   │  │  cada token solo mira tokens anteriores
│  │ + Residual    │  │
│  │ LayerNorm     │  │
│  │ FFN           │  │  dos capas lineales con GELU
│  │ + Residual    │  │
│  └───────────────┘  │
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│  LayerNorm final    │
│  Linear (→ vocab)   │  proyección a vocab_size dimensiones
│  Softmax            │  → probabilidades sobre el vocabulario
└─────────┬───────────┘
          │
          ▼
Output: prob("tejado") = 0.12, prob("piso") = 0.08, ...
```

### ¿Por qué "decoder-only"?

El Transformer original (Vaswani et al., 2017) tenía un **encoder** y un **decoder**. El encoder procesaba el input y el decoder generaba el output. GPT se queda **solo con el decoder** y le da una vuelta de tuerca genial:

- No hay encoder ni decoder separados
- El modelo recibe una secuencia de tokens y predice el siguiente
- La **máscara causal** asegura que cada token solo pueda "mirar" a los tokens anteriores (no al futuro)
- Entrenamiento: se predice el siguiente token para **cada posición** de la secuencia (teacher forcing)

Es como si le taparas todo lo que viene después y le preguntaras "¿qué sigue?" en cada posición.

### Los hiperparámetros clave

| Hiperparámetro | Qué controla | GPT-2 Small | Nuestro mini-GPT |
|:---------------|:-------------|:------------|:------------------|
| `vocab_size` | Tamaño del vocabulario | 50,257 | ~65 (char-level) |
| `d_model` | Dimensión de los embeddings | 768 | 64 |
| `n_heads` | Cabezas de attention | 12 | 4 |
| `n_layers` | Bloques transformer apilados | 12 | 4 |
| `block_size` | Longitud máxima de contexto | 1024 | 128 |
| `d_ff` | Dimensión interna del FFN | 3072 (4×768) | 256 (4×64) |

Nosotros vamos a construir un mini-GPT mucho más chiquito, pero con **exactamente la misma arquitectura**. La diferencia es solo de escala.

In [ ]:
# Hyperparameters for our mini-GPT
# We keep it small so it trains in minutes on CPU

block_size = 128       # max context length (how many tokens the model can see)
d_model = 64           # embedding dimension
n_heads = 4            # number of attention heads
n_layers = 4           # number of transformer blocks
dropout = 0.1          # dropout rate
learning_rate = 3e-4   # AdamW learning rate
batch_size = 32        # batch size
max_iters = 3000       # training iterations
eval_interval = 300    # how often to evaluate
eval_iters = 100       # batches for evaluation

print("Hyperparameters:")
print(f"  block_size  = {block_size}")
print(f"  d_model     = {d_model}")
print(f"  n_heads     = {n_heads}")
print(f"  n_layers    = {n_layers}")
print(f"  head_dim    = {d_model // n_heads}")
print(f"  d_ff        = {4 * d_model}")
print(f"  dropout     = {dropout}")

---

<a id='tokenizacion'></a>
## 2. Tokenización

Lo primero que necesitamos es convertir texto en números. Los modelos no entienden letras — entienden vectores numéricos. El proceso de convertir texto en una secuencia de enteros se llama **tokenización**.

### Character-level tokenizer

El tokenizer más simple posible: cada carácter es un token. Si nuestro texto tiene 65 caracteres únicos (letras, espacios, puntuación), entonces el vocabulario tiene 65 tokens.

**Ventajas:**
- Ultra simple de implementar
- Vocabulario muy chico
- Puede representar cualquier texto (nunca hay tokens desconocidos)

**Desventajas:**
- Secuencias muy largas (cada carácter es un token)
- El modelo tiene que aprender a "juntar letras en palabras" solo
- Ineficiente para producción

Para nuestra demo, character-level es perfecto porque mantiene todo simple.

### BPE (Byte Pair Encoding) — lo que usan los LLMs reales

Los modelos de producción (GPT-2, GPT-4, Claude, etc.) usan **BPE** o variantes. La idea es genial:

1. Empezás con un vocabulario de caracteres individuales (bytes)
2. Contás qué par de tokens adyacentes aparece más frecuentemente en el corpus
3. Fusionás ese par en un nuevo token
4. Repetís hasta tener el tamaño de vocabulario deseado (ej: 50,000 tokens)

Ejemplo de cómo BPE fusiona:
```
Paso 0: ['l', 'o', 'w', 'e', 'r']       → caracteres individuales
Paso 1: ['lo', 'w', 'e', 'r']            → 'l'+'o' era el par más frecuente
Paso 2: ['low', 'e', 'r']                → 'lo'+'w' es ahora el más frecuente
Paso 3: ['low', 'er']                    → 'e'+'r' fusionados
Paso 4: ['lower']                        → 'low'+'er' fusionados
```

El resultado es un vocabulario que tiene:
- Caracteres individuales (para palabras raras)
- Sub-palabras comunes ("ing", "tion", "pre")
- Palabras completas frecuentes ("the", "and")

Es un balance elegante entre vocabulario chico (char-level) y vocabulario enorme (word-level).

Nosotros vamos a implementar un character-level tokenizer para la demo, y mostrar cómo sería BPE.

In [ ]:
# --- Character-level tokenizer ---
# This is the simplest possible tokenizer: each character = one token

# Let's use a sample text to build our vocabulary
sample_text = "Hello, World! This is a sample text for our tokenizer."

# Build vocabulary from unique characters
chars = sorted(list(set(sample_text)))
vocab_size = len(chars)

# Create mappings: character <-> integer
stoi = {ch: i for i, ch in enumerate(chars)}  # string to int
itos = {i: ch for i, ch in enumerate(chars)}   # int to string

# Encode: string -> list of integers
def encode(text):
    return [stoi[ch] for ch in text]

# Decode: list of integers -> string
def decode(tokens):
    return ''.join([itos[t] for t in tokens])

# Demo
print(f"Vocabulary ({vocab_size} chars): {''.join(chars)}")
print(f"\nEncoding 'Hello':")
encoded = encode('Hello')
print(f"  'Hello' → {encoded}")
print(f"  {encoded} → '{decode(encoded)}'")
print(f"\nFull example:")
full_encoded = encode(sample_text)
print(f"  '{sample_text[:30]}...' → {full_encoded[:15]}...")

In [ ]:
# --- BPE tokenizer (simplified illustration) ---
# This shows the core BPE algorithm. Real implementations (tiktoken, sentencepiece)
# are more optimized but follow the same logic.

def get_pair_counts(token_sequences):
    """Count frequency of adjacent token pairs."""
    counts = {}
    for seq in token_sequences:
        for i in range(len(seq) - 1):
            pair = (seq[i], seq[i + 1])
            counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge_pair(token_sequences, pair, new_token):
    """Replace all occurrences of pair with new_token."""
    new_sequences = []
    for seq in token_sequences:
        new_seq = []
        i = 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair:
                new_seq.append(new_token)
                i += 2
            else:
                new_seq.append(seq[i])
                i += 1
        new_sequences.append(new_seq)
    return new_sequences

# Demo BPE on a small corpus
corpus = ["low", "lower", "lowest", "newer", "new", "wider"]
print("BPE merging process:")
print(f"Corpus: {corpus}")

# Start: each word is a list of characters
sequences = [list(word) for word in corpus]
vocab = list(set(ch for word in corpus for ch in word))
next_token_id = len(vocab)
merge_log = []

print(f"\nInitial vocab ({len(vocab)}): {sorted(vocab)}")
print(f"Initial sequences: {sequences}")

# Perform 5 merge steps
for step in range(5):
    counts = get_pair_counts(sequences)
    if not counts:
        break
    best_pair = max(counts, key=counts.get)
    new_token = best_pair[0] + best_pair[1]
    sequences = merge_pair(sequences, best_pair, new_token)
    vocab.append(new_token)
    merge_log.append((best_pair, new_token, counts[best_pair]))
    print(f"\nStep {step + 1}: merge '{best_pair[0]}' + '{best_pair[1]}' → '{new_token}' (appeared {counts[best_pair]}x)")
    print(f"  Sequences: {sequences}")

print(f"\nFinal vocab ({len(vocab)}): {sorted(vocab, key=len)}")
print("\nNotice: common subwords like 'er', 'ew', 'low' get their own tokens.")
print("This is how GPT-2's tokenizer was built — but with 50,257 merges on a huge corpus.")

---

<a id='embeddings'></a>
## 3. Embeddings

Ya tenemos tokens (enteros). Pero el modelo necesita **vectores continuos**, no enteros discretos. Para eso están los embeddings.

### Token embedding

Es una **tabla de lookup**: una matriz de tamaño `(vocab_size, d_model)`. Cada fila es el vector de un token. Cuando el token 42 entra al modelo, simplemente buscamos la fila 42 de la tabla.

$$\mathbf{e}_i = \mathbf{W}_E[\text{token}_i]$$

Donde $\mathbf{W}_E \in \mathbb{R}^{V \times d}$ es la **embedding matrix** con $V$ = vocab_size y $d$ = d_model.

Estos embeddings **se aprenden** durante el entrenamiento. Al principio son random, pero el modelo va ajustándolos para que tokens con significados similares tengan vectores cercanos.

### Positional embedding

Attention es **permutation-equivariant**: no tiene noción inherente de orden. "El gato come pescado" y "Pescado come el gato" producirían los mismos resultados si no agregamos información de posición.

GPT usa **positional embeddings aprendibles**: otra tabla de lookup de tamaño `(block_size, d_model)`. La posición 0 tiene un vector, la posición 1 otro, etc.

$$\mathbf{p}_i = \mathbf{W}_P[i]$$

Donde $\mathbf{W}_P \in \mathbb{R}^{T \times d}$ con $T$ = block_size (longitud máxima de contexto).

### La entrada al transformer

Los dos embeddings simplemente **se suman**:

$$\mathbf{x}_i = \mathbf{W}_E[\text{token}_i] + \mathbf{W}_P[i]$$

¿Por qué sumar y no concatenar? Porque concatenar duplicaría la dimensión y haría todo más costoso. Sumar funciona sorprendentemente bien porque los dos embeddings viven en el mismo espacio $\mathbb{R}^d$ y el modelo aprende a separar la información semántica de la posicional.

**Nota sobre el Transformer original:** Vaswani et al. usaban funciones sinusoidales fijas para los positional encodings. GPT usa embeddings aprendibles, lo cual es más flexible y en la práctica funciona igual o mejor.

In [ ]:
# --- Token and positional embeddings ---

# Example with a tiny vocabulary
demo_vocab_size = 10
demo_d_model = 8
demo_block_size = 6

# Token embedding table: each token ID maps to a d_model-dimensional vector
token_embedding = nn.Embedding(demo_vocab_size, demo_d_model)

# Positional embedding table: each position maps to a d_model-dimensional vector
position_embedding = nn.Embedding(demo_block_size, demo_d_model)

# Input: a batch of token sequences
tokens = torch.tensor([[2, 5, 8, 1, 7, 3]])  # (batch=1, seq_len=6)

# Lookup token embeddings
tok_emb = token_embedding(tokens)  # (1, 6, 8)

# Lookup positional embeddings (positions 0, 1, 2, 3, 4, 5)
positions = torch.arange(tokens.shape[1])  # [0, 1, 2, 3, 4, 5]
pos_emb = position_embedding(positions)     # (6, 8)

# Sum them
x = tok_emb + pos_emb  # (1, 6, 8) — broadcasting adds pos_emb to each batch

print("Token embedding table shape:", token_embedding.weight.shape)
print(f"  → {demo_vocab_size} tokens, each is a vector of size {demo_d_model}")
print(f"\nPosition embedding table shape: {position_embedding.weight.shape}")
print(f"  → {demo_block_size} positions, each is a vector of size {demo_d_model}")
print(f"\nInput tokens shape: {tokens.shape}")
print(f"Token embeddings shape: {tok_emb.shape}")
print(f"Position embeddings shape: {pos_emb.shape}")
print(f"Combined (tok + pos) shape: {x.shape}")
print(f"\nToken 2 embedding: {tok_emb[0, 0, :4].detach().numpy().round(3)}...")
print(f"Position 0 embedding: {pos_emb[0, :4].detach().numpy().round(3)}...")
print(f"Combined (token 2 @ pos 0): {x[0, 0, :4].detach().numpy().round(3)}...")
print(f"\nParameters: {demo_vocab_size * demo_d_model + demo_block_size * demo_d_model:,} "
      f"({demo_vocab_size * demo_d_model} token + {demo_block_size * demo_d_model} position)")

In [ ]:
# Visualize how token + position embeddings combine

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

tok_data = tok_emb[0].detach().numpy()
pos_data = pos_emb.detach().numpy()
combined = x[0].detach().numpy()

for ax, data, title in zip(axes, [tok_data, pos_data, combined],
                            ['Token Embeddings', 'Position Embeddings', 'Combined (sum)']):
    im = ax.imshow(data, cmap='RdBu', aspect='auto', vmin=-2, vmax=2)
    ax.set_xlabel('Embedding dimension')
    ax.set_ylabel('Position')
    ax.set_title(title)
    ax.set_yticks(range(6))
    ax.set_yticklabels([f'pos {i} (tok {tokens[0, i].item()})' for i in range(6)])

plt.colorbar(im, ax=axes, shrink=0.8)
plt.suptitle('Embeddings: token identity + positional info', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Each row is the input vector for one position.")
print("The model receives BOTH 'what token is here' and 'where in the sequence it is'.")

---

<a id='causal-attention'></a>
## 4. Causal self-attention

Esta es la pieza central del GPT. Ya vimos self-attention en el notebook anterior — cada token mira a todos los otros tokens y decide a cuáles prestar atención. Pero GPT tiene un requisito especial: **los tokens no pueden mirar al futuro**.

¿Por qué? Porque estamos entrenando el modelo para **predecir el siguiente token**. Si el token en la posición 5 pudiera mirar al token en la posición 6, la tarea sería trivial — simplemente copiaría la respuesta. No aprendería nada.

### La máscara causal

La solución es elegante: una **máscara triangular inferior** que pone $-\infty$ en las posiciones que corresponden al futuro. Cuando esos $-\infty$ pasan por softmax, se convierten en 0 — efectivamente ignorando esos tokens.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + \text{mask}\right) V$$

Donde la máscara es:

$$\text{mask}_{ij} = \begin{cases} 0 & \text{si } j \leq i \quad \text{(puede ver)} \\ -\infty & \text{si } j > i \quad \text{(futuro, bloqueado)} \end{cases}$$

Visualmente:
```
           Keys (tokens que puedo ver)
           t0    t1    t2    t3    t4
Queries  ┌─────┬─────┬─────┬─────┬─────┐
 t0      │  ✓  │ -∞  │ -∞  │ -∞  │ -∞  │  ← t0 solo se ve a sí mismo
 t1      │  ✓  │  ✓  │ -∞  │ -∞  │ -∞  │  ← t1 ve t0 y a sí mismo
 t2      │  ✓  │  ✓  │  ✓  │ -∞  │ -∞  │  ← t2 ve t0, t1 y a sí mismo
 t3      │  ✓  │  ✓  │  ✓  │  ✓  │ -∞  │
 t4      │  ✓  │  ✓  │  ✓  │  ✓  │  ✓  │  ← t4 ve todo
         └─────┴─────┴─────┴─────┴─────┘
```

### Multi-head attention

En vez de hacer una sola operación de attention con d_model dimensiones, dividimos en `n_heads` cabezas independientes, cada una con `d_k = d_model / n_heads` dimensiones.

¿Por qué? Porque cada cabeza puede aprender a prestar atención a **diferentes tipos de relaciones**:
- Una cabeza puede aprender relaciones sintácticas (sujeto-verbo)
- Otra puede aprender relaciones semánticas (sinónimos)
- Otra puede aprender posición relativa ("la palabra 2 posiciones atrás")

Cada cabeza tiene sus propias matrices $W_Q$, $W_K$, $W_V$ de tamaño `(d_model, d_k)`, y al final concatenamos las salidas de todas las cabezas y las proyectamos de vuelta a d_model.

En la práctica, en vez de hacer `n_heads` operaciones separadas, hacemos **una sola multiplicación matricial grande** y después reshapeamos. Es más eficiente en GPU.

In [ ]:
# --- The causal mask ---
# Let's see exactly what it does

T = 6  # sequence length

# Create causal mask: lower triangular = True (can attend), upper = False (future)
causal_mask = torch.tril(torch.ones(T, T))
print("Causal mask (1 = can see, 0 = blocked):")
print(causal_mask.int().numpy())

# In practice, we convert to -inf for the blocked positions
attn_mask = causal_mask.float().masked_fill(causal_mask == 0, float('-inf')).masked_fill(causal_mask == 1, 0.0)
print("\nAttention mask (0 = can see, -inf = blocked):")
print(attn_mask.numpy())

# Show what happens after softmax
# Fake attention scores (before mask)
fake_scores = torch.randn(T, T)
print("\nRaw attention scores (before mask):")
print(fake_scores.numpy().round(2))

# Apply mask and softmax
masked_scores = fake_scores + attn_mask
attention_weights = F.softmax(masked_scores, dim=-1)
print("\nAttention weights (after mask + softmax):")
print(attention_weights.numpy().round(3))
print("\nNote: each row sums to 1, and future positions have weight 0.")
print(f"Row sums: {attention_weights.sum(dim=-1).numpy().round(3)}")

In [ ]:
# --- Causal self-attention from scratch ---

class CausalSelfAttention(nn.Module):
    """
    Multi-head causal self-attention.
    
    Each token can only attend to previous tokens (and itself).
    We compute Q, K, V for all heads in a single matrix multiply for efficiency.
    """
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        
        # Q, K, V projections (all in one matrix for efficiency)
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        
        # Output projection
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        # Dropout
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        
        # Causal mask — registered as buffer (not a parameter, but moves to GPU with model)
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        )
    
    def forward(self, x):
        B, T, C = x.shape  # batch, seq_len, d_model
        
        # Compute Q, K, V in one shot
        qkv = self.qkv_proj(x)  # (B, T, 3*d_model)
        q, k, v = qkv.chunk(3, dim=-1)  # each is (B, T, d_model)
        
        # Reshape to (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        
        # Attention scores: (B, n_heads, T, T)
        scale = 1.0 / math.sqrt(self.head_dim)
        attn = (q @ k.transpose(-2, -1)) * scale
        
        # Apply causal mask: set future positions to -inf
        attn = attn.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        
        # Softmax → attention weights
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_dropout(attn)
        
        # Apply attention to values
        out = attn @ v  # (B, n_heads, T, head_dim)
        
        # Concatenate heads: (B, T, d_model)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        
        # Output projection
        out = self.resid_dropout(self.out_proj(out))
        
        return out

# Test it
attn_module = CausalSelfAttention(d_model=64, n_heads=4, block_size=128)
test_input = torch.randn(2, 10, 64)  # batch=2, seq_len=10, d_model=64
test_output = attn_module(test_input)

print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_output.shape}")
print(f"\nParameters: {sum(p.numel() for p in attn_module.parameters()):,}")
print(f"  qkv_proj: {64 * 3 * 64:,} (3 × d_model × d_model, for Q, K, V)")
print(f"  out_proj: {64 * 64:,} (d_model × d_model)")

In [ ]:
# Visualize the causal attention pattern
# Let's see what attention weights look like with random weights (before training)

with torch.no_grad():
    # Run a forward pass and capture attention weights
    B, T, C = 1, 8, 64
    x = torch.randn(B, T, C)
    
    # Manually compute attention to get the weights
    qkv = attn_module.qkv_proj(x)
    q, k, v = qkv.chunk(3, dim=-1)
    q = q.view(B, T, 4, 16).transpose(1, 2)
    k = k.view(B, T, 4, 16).transpose(1, 2)
    
    scale = 1.0 / math.sqrt(16)
    attn_scores = (q @ k.transpose(-2, -1)) * scale
    mask = torch.tril(torch.ones(T, T)).view(1, 1, T, T)
    attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(attn_scores, dim=-1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for head in range(4):
    ax = axes[head]
    w = attn_weights[0, head].numpy()
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Head {head}')
    ax.set_xlabel('Key position')
    if head == 0:
        ax.set_ylabel('Query position')
    ax.set_xticks(range(T))
    ax.set_yticks(range(T))

plt.colorbar(im, ax=axes, shrink=0.8)
plt.suptitle('Causal attention weights (random init)\nNotice: upper triangle is always 0 (cannot see future)', fontsize=12)
plt.tight_layout()
plt.show()

print("The triangular pattern is the causal mask in action.")
print("Each row shows which previous tokens a given position attends to.")
print("After training, these patterns become meaningful (syntax, coreference, etc.).")

---

<a id='ffn'></a>
## 5. Feed-forward network

Después de la attention, cada posición pasa por una **red feed-forward** idéntica e independiente. Es sorprendentemente simple:

$$\text{FFN}(x) = \text{GELU}(x W_1 + b_1) W_2 + b_2$$

Donde:
- $W_1 \in \mathbb{R}^{d_{model} \times d_{ff}}$ — expande de d_model a d_ff (típicamente 4× d_model)
- $W_2 \in \mathbb{R}^{d_{ff} \times d_{model}}$ — comprime de vuelta a d_model

### ¿Por qué expandir y comprimir?

La expansión a 4× es como darle al modelo un "espacio de trabajo" más grande para hacer cálculos intermedios. Imaginá que attention mezcla información entre posiciones, y FFN **procesa** esa información en cada posición. La expansión le da más capacidad para transformaciones no lineales complejas.

Es como attention es la parte "social" (comunicación entre tokens) y FFN es la parte "individual" (procesamiento local de cada token).

### GELU (Gaussian Error Linear Unit)

$$\text{GELU}(x) = x \cdot \Phi(x)$$

Donde $\Phi(x)$ es la CDF de la distribución normal estándar. Es una versión suave de ReLU que no tiene la "esquina" en x=0. GPT-2 usa GELU en vez de ReLU porque:
- No tiene el problema de "neuronas muertas" de ReLU
- Permite gradientes pequeños pero no-zero para inputs negativos
- Funciona ligeramente mejor en la práctica para transformers

In [ ]:
# --- GELU activation ---

x = torch.linspace(-4, 4, 200)
relu = F.relu(x)
gelu = F.gelu(x)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x.numpy(), relu.numpy(), label='ReLU', linewidth=2, alpha=0.8)
ax.plot(x.numpy(), gelu.numpy(), label='GELU', linewidth=2, alpha=0.8)
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axvline(x=0, color='gray', linewidth=0.5)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('activation(x)', fontsize=12)
ax.set_title('GELU vs ReLU — GELU is smooth, no dead neurons', fontsize=13)
ax.legend(fontsize=12)
ax.set_xlim(-4, 4)
ax.set_ylim(-1, 4)
plt.tight_layout()
plt.show()

print("GELU is what GPT-2 and most modern transformers use.")
print("For negative x, GELU gives small negative values (not zero like ReLU).")
print("This prevents dead neurons and allows smoother optimization.")

In [ ]:
# --- Feed-forward network ---

class FeedForward(nn.Module):
    """
    Position-wise feed-forward network.
    Expand to 4x, apply GELU, compress back.
    """
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),   # expand
            nn.GELU(),                           # activation
            nn.Linear(4 * d_model, d_model),    # compress back
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)

# Test it
ffn = FeedForward(d_model=64)
test_input = torch.randn(2, 10, 64)  # batch=2, seq_len=10, d_model=64
test_output = ffn(test_input)

print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_output.shape}")
print(f"\nParameters:")
print(f"  W1: {64} × {4 * 64} = {64 * 4 * 64:,} weights + {4 * 64} bias")
print(f"  W2: {4 * 64} × {64} = {4 * 64 * 64:,} weights + {64} bias")
print(f"  Total: {sum(p.numel() for p in ffn.parameters()):,}")
print(f"\nExpansion ratio: d_model ({64}) → 4 × d_model ({4 * 64}) → d_model ({64})")
print("Each position is processed independently — no communication between positions.")

---

<a id='transformer-block'></a>
## 6. Transformer block

Un transformer block combina attention y FFN con **layer normalization** y **residual connections**. Es el "ladrillo" del GPT — se apila N veces.

### Pre-norm (GPT-2 style)

GPT-2 (y la mayoría de los modelos modernos) usa **pre-norm**: la normalización va **antes** de cada sub-capa, no después. Esto es diferente del transformer original (post-norm).

```
        x
        │
        ├──────────────────┐
        │                  │ (residual)
        ▼                  │
    LayerNorm              │
        │                  │
        ▼                  │
  Causal Attention         │
        │                  │
        ▼                  │
       (+) ◄──────────────┘
        │
        ├──────────────────┐
        │                  │ (residual)
        ▼                  │
    LayerNorm              │
        │                  │
        ▼                  │
      FFN                  │
        │                  │
        ▼                  │
       (+) ◄──────────────┘
        │
        ▼
      output
```

### ¿Por qué residual connections?

Sin residual connections, la señal tiene que pasar a través de TODAS las capas de attention y FFN. Con 12 capas (o 96 en GPT-3), los gradientes se degradan. Las residual connections proveen un "atajo" directo:

$$\text{output} = x + \text{Attention}(\text{LayerNorm}(x))$$

El gradiente puede fluir directamente por el `+`, sin degradarse. Es como tener una "autopista" para los gradientes.

### ¿Por qué LayerNorm?

Normaliza las activaciones para que tengan media 0 y varianza 1 en la dimensión de features. Esto estabiliza el entrenamiento y permite usar learning rates más altos.

$$\text{LayerNorm}(x) = \frac{x - \mu}{\sigma + \epsilon} \cdot \gamma + \beta$$

Donde $\gamma$ y $\beta$ son parámetros aprendibles (scale y shift).

In [ ]:
# --- Transformer block ---

class TransformerBlock(nn.Module):
    """
    One transformer block: LayerNorm → Attention → Residual → LayerNorm → FFN → Residual
    Uses pre-norm (GPT-2 style).
    """
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, dropout)
    
    def forward(self, x):
        # Pre-norm: normalize BEFORE the sub-layer
        x = x + self.attn(self.ln1(x))   # attention + residual
        x = x + self.ffn(self.ln2(x))    # FFN + residual
        return x

# Test it
block = TransformerBlock(d_model=64, n_heads=4, block_size=128)
test_input = torch.randn(2, 10, 64)
test_output = block(test_input)

print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_output.shape}")
print(f"\nSame shape in = same shape out. That's what allows stacking.")
print(f"\nParameters per block: {sum(p.numel() for p in block.parameters()):,}")
print(f"  LayerNorm 1: {2 * 64:,} (gamma + beta)")
print(f"  Attention:   {sum(p.numel() for p in block.attn.parameters()):,}")
print(f"  LayerNorm 2: {2 * 64:,}")
print(f"  FFN:         {sum(p.numel() for p in block.ffn.parameters()):,}")

In [ ]:
# Verify the residual connection is working
# Without residuals, deep networks lose information rapidly

def measure_signal_preservation(n_layers, d_model=64, n_heads=4, use_residual=True):
    """Measure how well the signal is preserved through layers."""
    torch.manual_seed(42)
    x = torch.randn(1, 10, d_model)
    original_norm = x.norm().item()
    norms = [original_norm]
    
    for i in range(n_layers):
        block = TransformerBlock(d_model, n_heads, 128, dropout=0.0)
        with torch.no_grad():
            if use_residual:
                x = block(x)
            else:
                # Without residual: just attention + FFN, no skip connection
                h = block.attn(block.ln1(x))
                h = block.ffn(block.ln2(h))
                x = h
        norms.append(x.norm().item())
    return norms

fig, ax = plt.subplots(figsize=(9, 5))
n_layers_test = 20

norms_with = measure_signal_preservation(n_layers_test, use_residual=True)
norms_without = measure_signal_preservation(n_layers_test, use_residual=False)

ax.plot(range(n_layers_test + 1), norms_with, 'o-', label='With residuals', linewidth=2, markersize=5)
ax.plot(range(n_layers_test + 1), norms_without, 's-', label='Without residuals', linewidth=2, markersize=5)
ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('Signal norm', fontsize=12)
ax.set_title('Residual connections preserve signal through deep networks', fontsize=13)
ax.legend(fontsize=11)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print(f"After {n_layers_test} layers:")
print(f"  With residuals:    norm = {norms_with[-1]:.4f}")
print(f"  Without residuals: norm = {norms_without[-1]:.6f}")
print("\nResidual connections prevent the signal from vanishing or exploding.")

---

<a id='implementacion'></a>
## 7. Implementación completa del GPT

Ahora juntamos todo en una sola clase. El GPT completo es simplemente:

1. **Token embedding** + **positional embedding** → se suman
2. **N transformer blocks** apilados
3. **LayerNorm** final
4. **Linear projection** al tamaño del vocabulario → logits

Los logits pasan por softmax para obtener probabilidades sobre el vocabulario. Durante el entrenamiento, usamos cross-entropy loss entre los logits y los tokens target (el siguiente token en cada posición).

### ¿Cómo funciona el entrenamiento?

Dado un texto "El gato se sentó":

```
Input:  [El]  [gato] [se]  [sentó]
Target: [gato] [se]  [sentó] [en]
```

En cada posición, el modelo predice el **siguiente** token. Gracias a la máscara causal, cada posición solo ve los tokens anteriores, así que cada posición es un ejemplo de entrenamiento independiente. Con un contexto de 128 tokens, cada secuencia nos da 128 ejemplos de entrenamiento.

La loss es:

$$\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T} \log P(x_t | x_{<t})$$

Es decir, cross-entropy promediando sobre todas las posiciones.

In [ ]:
# --- Full GPT model ---

class GPT(nn.Module):
    """
    A complete GPT (decoder-only transformer) language model.
    
    Architecture:
        Token embedding + positional embedding
        → N transformer blocks (LayerNorm → CausalAttention → Residual → LayerNorm → FFN → Residual)
        → LayerNorm → Linear projection to vocab
    """
    def __init__(self, vocab_size, d_model, n_heads, n_layers, block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        
        # Embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Transformer blocks
        self.blocks = nn.Sequential(*[
            TransformerBlock(d_model, n_heads, block_size, dropout)
            for _ in range(n_layers)
        ])
        
        # Final layer norm (pre-norm style: one more LN at the end)
        self.ln_final = nn.LayerNorm(d_model)
        
        # Output projection: d_model → vocab_size
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying: share weights between token embedding and output projection
        # This is a common trick that reduces parameters and improves performance
        self.token_embedding.weight = self.lm_head.weight
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        """Initialize weights with small random values."""
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)
    
    def forward(self, idx, targets=None):
        """
        Forward pass.
        
        Args:
            idx: (B, T) tensor of token indices
            targets: (B, T) tensor of target token indices (optional, for loss)
        
        Returns:
            logits: (B, T, vocab_size) raw scores for each token in vocabulary
            loss: scalar loss (only if targets provided)
        """
        B, T = idx.shape
        assert T <= self.block_size, f"Sequence length {T} exceeds block_size {self.block_size}"
        
        # Token + positional embeddings
        tok_emb = self.token_embedding(idx)                    # (B, T, d_model)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))  # (T, d_model)
        x = self.dropout(tok_emb + pos_emb)                    # (B, T, d_model)
        
        # Pass through transformer blocks
        x = self.blocks(x)                                     # (B, T, d_model)
        
        # Final layer norm + projection to vocabulary
        x = self.ln_final(x)                                   # (B, T, d_model)
        logits = self.lm_head(x)                               # (B, T, vocab_size)
        
        # Compute loss if targets are provided
        loss = None
        if targets is not None:
            # Flatten: (B*T, vocab_size) vs (B*T,)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        
        return logits, loss
    
    def count_parameters(self):
        """Count total and per-component parameters."""
        total = sum(p.numel() for p in self.parameters())
        # Note: token_embedding and lm_head share weights, so we count them once
        embedding = sum(p.numel() for p in self.token_embedding.parameters())
        position = sum(p.numel() for p in self.position_embedding.parameters())
        transformer = sum(p.numel() for p in self.blocks.parameters())
        ln_final = sum(p.numel() for p in self.ln_final.parameters())
        return {
            'total': total,
            'token_embedding (shared with lm_head)': embedding,
            'position_embedding': position,
            'transformer_blocks': transformer,
            'ln_final': ln_final,
        }

# Test with a dummy vocabulary
test_model = GPT(
    vocab_size=100,
    d_model=64,
    n_heads=4,
    n_layers=4,
    block_size=128,
    dropout=0.1
)

# Forward pass
dummy_input = torch.randint(0, 100, (2, 32))     # batch=2, seq_len=32
dummy_target = torch.randint(0, 100, (2, 32))    # targets
logits, loss = test_model(dummy_input, dummy_target)

print(f"Model output:")
print(f"  Logits shape: {logits.shape}  (batch, seq_len, vocab_size)")
print(f"  Loss: {loss.item():.4f}")
print(f"  Expected initial loss: {-np.log(1/100):.4f} (= -log(1/vocab_size), random guessing)")
print(f"\nParameter counts:")
for name, count in test_model.count_parameters().items():
    print(f"  {name}: {count:,}")

---

<a id='training'></a>
## 8. Training loop

Vamos a entrenar nuestro GPT en **Tiny Shakespeare** — un dataset clásico para demos de modelos de lenguaje. Son ~1MB de texto de obras de Shakespeare. Es suficiente para que el modelo aprenda patrones de inglés, nombres de personajes, y estructura de diálogos.

### El dataset

- ~1 millón de caracteres
- ~65 caracteres únicos (vocabulario)
- El modelo aprende a generar texto estilo Shakespeare

### Preparación de datos

Para language modeling, la preparación es muy simple:
1. Tokenizamos todo el texto en una secuencia larga de enteros
2. Para cada batch, cortamos chunks aleatorios de longitud `block_size`
3. Los targets son simplemente los mismos chunks shifted una posición a la derecha

```
Texto:    "First Citizen:\nBefore we proceed"
Tokens:   [18, 47, 56, 57, 58, 1, 15, 47, ...]
Input:    [18, 47, 56, 57, 58, 1, 15, 47]    ← posiciones 0..7
Target:   [47, 56, 57, 58, 1, 15, 47, 58]    ← posiciones 1..8
```

In [ ]:
# --- Load Tiny Shakespeare ---
import urllib.request
import os

data_path = 'tiny_shakespeare.txt'
if not os.path.exists(data_path):
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    print(f"Downloading Tiny Shakespeare from {url}...")
    urllib.request.urlretrieve(url, data_path)
    print("Done!")

with open(data_path, 'r') as f:
    text = f.read()

print(f"Dataset size: {len(text):,} characters")
print(f"First 200 characters:")
print(text[:200])

In [ ]:
# --- Build tokenizer from the dataset ---

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars)}")
print(f"\nExample:")
print(f"  encode('hello') = {encode('hello')}")
print(f"  decode([46, 43, 50, 50, 53]) = '{decode([46, 43, 50, 50, 53])}'")

In [ ]:
# --- Prepare train/val split and data loading ---

# Encode the entire dataset
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Encoded dataset: {data.shape[0]:,} tokens")
print(f"First 20 tokens: {data[:20].tolist()}")
print(f"Decoded back: '{decode(data[:20].tolist())}'")

# Train/val split (90/10)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"\nTrain: {len(train_data):,} tokens")
print(f"Val:   {len(val_data):,} tokens")

def get_batch(split):
    """Get a random batch of (input, target) pairs."""
    data_split = train_data if split == 'train' else val_data
    # Random starting indices
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    # Stack chunks into a batch
    x = torch.stack([data_split[i:i + block_size] for i in ix])
    y = torch.stack([data_split[i + 1:i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# Demo: one batch
xb, yb = get_batch('train')
print(f"\nBatch shapes: input {xb.shape}, target {yb.shape}")
print(f"\nExample (first sequence, first 10 tokens):")
print(f"  Input:  {xb[0, :10].tolist()} → '{decode(xb[0, :10].tolist())}'")
print(f"  Target: {yb[0, :10].tolist()} → '{decode(yb[0, :10].tolist())}'")
print(f"\nNotice: target is just input shifted by one position.")

In [ ]:
# --- Create the model ---

model = GPT(
    vocab_size=vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    block_size=block_size,
    dropout=dropout
).to(device)

param_counts = model.count_parameters()
print("GPT Model Summary:")
print(f"{'=' * 50}")
for name, count in param_counts.items():
    print(f"  {name}: {count:,}")
print(f"{'=' * 50}")
print(f"\nFor reference:")
print(f"  Our mini-GPT:  {param_counts['total']:>12,} parameters")
print(f"  GPT-2 Small:   124,000,000 parameters")
print(f"  GPT-2 XL:    1,500,000,000 parameters")
print(f"  GPT-3:     175,000,000,000 parameters")

In [ ]:
# --- Training loop ---

@torch.no_grad()
def estimate_loss(model):
    """Estimate loss on train and val sets."""
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# Simple generation function for evolution tracking
@torch.no_grad()
def generate_sample(model, prompt="\n", max_new_tokens=100, temperature=0.8):
    """Quick generation for tracking evolution."""
    model.eval()
    tokens = encode(prompt)
    idx = torch.tensor([tokens], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)
    return decode(idx[0].tolist())

# Optimizer: AdamW (Adam with decoupled weight decay)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training!
print(f"Training for {max_iters} iterations...")
print(f"Using device: {device}")
print()

train_losses = []
val_losses = []
log_steps = []
generated_samples = []  # Track generated text evolution

start_time = time.time()

for step in range(max_iters):
    # Evaluate periodically
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss(model)
        elapsed = time.time() - start_time
        
        # Generate a sample to track evolution
        sample = generate_sample(model, prompt="\n", max_new_tokens=150, temperature=0.8)
        generated_samples.append((step, sample))
        
        print(f"  step {step:>5d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f} | {elapsed:.1f}s")
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        log_steps.append(step)
    
    # Get batch
    xb, yb = get_batch('train')
    
    # Forward pass
    logits, loss = model(xb, yb)
    
    # Backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

total_time = time.time() - start_time
print(f"\nTraining complete! Total time: {total_time:.1f}s")
print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final val loss:   {val_losses[-1]:.4f}")

In [ ]:
# --- Show evolution of generated text during training ---

print("=" * 70)
print("EVOLUTION OF GENERATED TEXT DURING TRAINING")
print("=" * 70)
print("\nWatch how the model learns to generate coherent text!")
print("Early: random characters. Later: words, phrases, structure.\n")

for step, sample in generated_samples:
    print(f"\n{'─' * 70}")
    print(f"Step {step:>5d} (loss: {train_losses[log_steps.index(step)]:.4f})")
    print(f"{'─' * 70}")
    # Show first 200 chars of generated text
    display_text = sample[:200] if len(sample) > 200 else sample
    print(display_text)
    if len(sample) > 200:
        print("...")

print(f"\n{'─' * 70}")
print("\nNotice the progression:")
print("  • Step 0:     Random characters, no structure")
print("  • Step 300:   Some letter patterns emerge")
print("  • Step 600:   Words start appearing")
print("  • Step 900+:  Coherent phrases, punctuation, structure")
print("\nThe model learns language patterns gradually, just like a human!")

In [ ]:
# --- Plot training curves ---

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(log_steps, train_losses, 'o-', label='Train loss', markersize=5, linewidth=2)
ax.plot(log_steps, val_losses, 's-', label='Val loss', markersize=5, linewidth=2)

# Add random baseline
random_loss = -np.log(1 / vocab_size)
ax.axhline(y=random_loss, color='gray', linestyle='--', alpha=0.7, label=f'Random baseline ({random_loss:.2f})')

ax.set_xlabel('Step', fontsize=12)
ax.set_ylabel('Loss (cross-entropy)', fontsize=12)
ax.set_title('GPT Training — Loss over time', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Random guessing loss: {random_loss:.4f} (= -log(1/{vocab_size}))")
print(f"Our model's val loss: {val_losses[-1]:.4f}")
print(f"\nThe model is MUCH better than random — it has learned patterns in the text.")

---

<a id='generacion'></a>
## 9. Generación de texto

Ahora viene la parte divertida: **generar texto**. El proceso es autoregresivo:

1. Le damos al modelo un prompt (secuencia inicial de tokens)
2. El modelo predice la distribución de probabilidad del siguiente token
3. **Sampleamos** un token de esa distribución
4. Lo agregamos a la secuencia
5. Repetimos

```
Prompt: "To be or"
                    ┌─────────────────────────────────┐
Step 1:  modelo(["To be or"]) → prob(" not") = 0.3   │
                                  sample → " not"     │
                                                      │ loop
Step 2:  modelo(["To be or not"]) → prob(" to") = 0.4│
                                     sample → " to"  │
                                                      │
Step 3:  modelo(["To be or not to"]) → ...            │
                    └─────────────────────────────────┘
```

### ¿Cómo sampleamos?

Hay varias estrategias:

**Greedy (argmax):** Siempre elegir el token más probable. Produce texto repetitivo y aburrido.

**Temperature sampling:** Dividir los logits por un valor $\tau$ (temperature) antes del softmax:

$$P(x_i) = \frac{e^{z_i / \tau}}{\sum_j e^{z_j / \tau}}$$

- $\tau = 1.0$: distribución original
- $\tau < 1.0$: más determinístico (picos más altos, los tokens probables se vuelven aún más probables)
- $\tau > 1.0$: más aleatorio (distribución más plana)
- $\tau \to 0$: equivale a greedy

**Top-k sampling:** Antes de samplear, quedarse solo con los top-k tokens más probables y redistribuir la masa de probabilidad entre ellos. Esto evita samplear tokens muy improbables (que generarían basura).

En la práctica, los modelos buenos usan una combinación de temperature + top-k (o top-p / nucleus sampling).

In [ ]:
# --- Text generation ---

@torch.no_grad()
def generate(model, prompt, max_new_tokens=200, temperature=1.0, top_k=None):
    """
    Autoregressive text generation.
    
    Args:
        model: the GPT model
        prompt: string to start generation from
        max_new_tokens: how many tokens to generate
        temperature: sampling temperature (lower = more deterministic)
        top_k: if set, only sample from top-k most likely tokens
    """
    model.eval()
    
    # Encode the prompt
    tokens = encode(prompt)
    idx = torch.tensor([tokens], dtype=torch.long, device=device)  # (1, T)
    
    for _ in range(max_new_tokens):
        # Crop to block_size if needed (sliding window)
        idx_cond = idx[:, -model.block_size:]
        
        # Forward pass → logits for all positions
        logits, _ = model(idx_cond)
        
        # We only care about the last position (next token prediction)
        logits = logits[:, -1, :]  # (1, vocab_size)
        
        # Apply temperature
        if temperature != 1.0:
            logits = logits / temperature
        
        # Apply top-k filtering
        if top_k is not None:
            # Keep only top-k logits, set rest to -inf
            top_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < top_values[:, [-1]]] = float('-inf')
        
        # Convert to probabilities
        probs = F.softmax(logits, dim=-1)  # (1, vocab_size)
        
        # Sample from the distribution
        next_token = torch.multinomial(probs, num_samples=1)  # (1, 1)
        
        # Append to the sequence
        idx = torch.cat([idx, next_token], dim=1)
    
    # Decode the full sequence
    return decode(idx[0].tolist())

# Generate some text!
print("="  * 60)
print("GENERATED TEXT (temperature=0.8, top_k=40)")
print("=" * 60)
generated = generate(model, prompt="\n", max_new_tokens=500, temperature=0.8, top_k=40)
print(generated)
print("=" * 60)

In [ ]:
# --- Compare different sampling strategies ---

prompt = "ROMEO:\n"

strategies = [
    {"name": "Greedy (temp=0.01)", "temperature": 0.01, "top_k": None},
    {"name": "Conservative (temp=0.5, top_k=10)", "temperature": 0.5, "top_k": 10},
    {"name": "Balanced (temp=0.8, top_k=40)", "temperature": 0.8, "top_k": 40},
    {"name": "Creative (temp=1.2, top_k=None)", "temperature": 1.2, "top_k": None},
]

for s in strategies:
    print(f"\n{'─' * 50}")
    print(f"Strategy: {s['name']}")
    print(f"{'─' * 50}")
    result = generate(
        model, prompt=prompt, max_new_tokens=200,
        temperature=s['temperature'], top_k=s['top_k']
    )
    # Print just the generated part (not the prompt)
    print(result[:300])

print(f"\n{'─' * 50}")
print("\nNotice the trade-off:")
print("  - Low temperature → repetitive but 'safe'")
print("  - High temperature → creative but sometimes incoherent")
print("  - top_k helps prevent very unlikely (garbage) tokens")

In [ ]:
# --- Visualize temperature effect on the probability distribution ---

# Get logits for a real position
model.eval()
with torch.no_grad():
    sample_tokens = torch.tensor([encode("To be or not to ")], device=device)
    logits_sample, _ = model(sample_tokens)
    last_logits = logits_sample[0, -1, :].cpu()  # logits for next token prediction

temperatures = [0.3, 0.7, 1.0, 1.5, 2.5]

fig, axes = plt.subplots(1, len(temperatures), figsize=(18, 4), sharey=True)

for ax, temp in zip(axes, temperatures):
    probs = F.softmax(last_logits / temp, dim=-1).numpy()
    top_indices = np.argsort(probs)[-15:][::-1]
    top_probs = probs[top_indices]
    top_chars = [itos[i] if itos[i].strip() else repr(itos[i]) for i in top_indices]
    
    colors = plt.cm.Blues(np.linspace(0.8, 0.3, len(top_indices)))
    ax.barh(range(len(top_indices)), top_probs, color=colors)
    ax.set_yticks(range(len(top_indices)))
    ax.set_yticklabels(top_chars, fontsize=9)
    ax.set_title(f'τ = {temp}', fontsize=12)
    ax.set_xlim(0, max(0.5, top_probs[0] * 1.2))
    ax.invert_yaxis()

axes[0].set_ylabel('Token')
fig.suptitle('Effect of temperature on next-token probabilities', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Low temperature → peaked distribution (model is confident, picks top token)")
print("High temperature → flat distribution (more random, more diverse)")

In [ ]:
# --- Visualize top-k sampling ---

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

probs_full = F.softmax(last_logits / 0.8, dim=-1).numpy()

for ax, k, title in zip(axes, [5, 20, vocab_size], ['Top-5', 'Top-20', f'Full (all {vocab_size})']):
    probs_k = probs_full.copy()
    if k < vocab_size:
        top_k_indices = np.argsort(probs_k)[-k:]
        mask = np.ones(vocab_size, dtype=bool)
        mask[top_k_indices] = False
        probs_k[mask] = 0
        probs_k = probs_k / probs_k.sum()  # re-normalize
    
    sorted_probs = np.sort(probs_k)[::-1]
    ax.bar(range(min(50, vocab_size)), sorted_probs[:50], color='steelblue', alpha=0.8)
    ax.set_title(f'{title}', fontsize=12)
    ax.set_xlabel('Token rank')
    ax.set_ylabel('Probability')
    non_zero = (probs_k > 0).sum()
    ax.text(0.95, 0.95, f'{non_zero} tokens\nwith P > 0',
            transform=ax.transAxes, ha='right', va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Top-k sampling: restricting the vocabulary to the k most likely tokens', fontsize=13)
plt.tight_layout()
plt.show()

print("Top-k removes the long tail of unlikely tokens.")
print("This prevents the model from generating garbage tokens")
print("while still allowing diversity among the top candidates.")

---

<a id='resumen'></a>
## 10. Resumen

### Lo que construimos

Construimos un GPT completo desde cero, pieza por pieza. Cada componente tiene un rol claro:

| Componente | Qué hace | Parámetros clave |
|:-----------|:---------|:-----------------|
| **Tokenización** | Texto → secuencia de enteros. Char-level para demo, BPE para producción | `vocab_size` |
| **Token Embedding** | Tabla de lookup: token ID → vector de d_model dimensiones | `vocab_size × d_model` |
| **Position Embedding** | Inyecta información de posición. Se suma al token embedding | `block_size × d_model` |
| **Causal Self-Attention** | Cada token asiste a tokens anteriores. Máscara triangular inferior. Multi-head para diversidad | `4 × d_model²` (QKV + out) |
| **Feed-Forward Network** | Procesamiento local por posición. Expande 4×, GELU, comprime | `8 × d_model²` + biases |
| **Transformer Block** | Pre-LayerNorm → Attention → Residual → Pre-LayerNorm → FFN → Residual | Attention + FFN + 2×LN |
| **Output Projection** | d_model → vocab_size logits. Softmax → probabilidades. Weight tying con token embedding | compartido |
| **Training** | Cross-entropy loss. AdamW optimizer. Teacher forcing (targets = input shifted by 1) | `lr`, `batch_size`, `iters` |
| **Generación** | Autoregresivo: predecir → samplear → agregar → repetir. Temperature y top-k controlan creatividad | `temperature`, `top_k` |

### Lo que hace a GPT especial

| Decisión de diseño | Por qué |
|:-------------------|:--------|
| **Decoder-only** | Más simple que encoder-decoder. Una sola tarea: predecir el siguiente token |
| **Causal mask** | Permite entrenar en paralelo (teacher forcing) sin hacer trampa |
| **Pre-norm** | Más estable que post-norm para redes profundas |
| **GELU** | No tiene neuronas muertas como ReLU |
| **Weight tying** | Reduce parámetros, mejora performance |
| **AdamW** | Adam con weight decay desacoplado, estándar para transformers |

### La escala importa

Nuestro mini-GPT tiene ~100K parámetros y genera texto que se parece vagamente a Shakespeare. Los GPTs reales son exactamente la misma arquitectura, pero mucho más grandes:

| Modelo | Parámetros | Datos de entrenamiento | Contexto |
|:-------|:-----------|:----------------------|:---------|
| Nuestro mini-GPT | ~100K | ~1MB (Shakespeare) | 128 tokens |
| GPT-2 Small | 124M | 40GB (WebText) | 1,024 tokens |
| GPT-3 | 175B | ~570GB | 2,048 tokens |
| GPT-4 | ??? (rumores: ~1.8T MoE) | ??? | 128K tokens |

La arquitectura es la misma. Lo que cambia es la escala — y con la escala vienen capacidades emergentes como razonamiento, traducción, y seguimiento de instrucciones.

### ¿Qué viene después?

Construir el GPT es solo el principio. Para hacerlo práctico necesitamos:

- **Eficiencia en GPU**: cómo paralelizar y distribuir el entrenamiento en múltiples GPUs
- **Mixed precision**: entrenar en fp16/bf16 para ir 2× más rápido con la mitad de memoria
- **Gradient accumulation**: simular batch sizes grandes sin necesitar tanta RAM
- **Flash attention**: attention optimizada que es mucho más rápida y usa menos memoria

---

**Siguiente notebook →** [13 - Eficiencia y GPUs](./13_eficiencia_gpus.ipynb): cómo escalar el entrenamiento — GPUs, mixed precision, gradient accumulation, y las técnicas que hacen posible entrenar modelos de miles de millones de parámetros.